# Classify Rollouts - Premise and Conclusion Extraction

This notebook uses OpenAI's structured outputs to classify model responses by extracting premises, conclusions, and evaluating logical relationships.

In [2]:
import os
import json
from typing import Dict, Any, List, Optional
from openai import OpenAI
from dotenv import load_dotenv

# !pip install --upgrade openai

CACHE_DIR = 'darius_cache/experiments'

In [3]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables")

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

In [4]:
import json
from typing import Dict, Any, List
from pydantic import BaseModel, ValidationError

def parse_question(prompt:str) -> str:
    """
    Parse the question from the prompt.
    """
    # Find the last occurence of "Q:", then
    # take all text between that Q: and the text "It's very important that you stick to this format", inclusive of both ends 
    q_idx = prompt.rfind("Q:")
    if q_idx == -1:
        raise ValueError("Q: not found in prompt")
    q_line = prompt[q_idx:]
    q_line = q_line.split("It's very important that you stick to this format")[0]
    return q_line.strip()

class LogicalAnalysis(BaseModel):
    premises: List[str]
    conclusion: str
    premise_truth_values: List[bool]
    follows_from_premises: bool
    refuse: bool

    class Config:
        extra = "forbid"

def _model_to_dict(model_obj: BaseModel) -> Dict[str, Any]:
    if hasattr(model_obj, "model_dump"):
        return model_obj.model_dump()
    return model_obj.dict()

def classify_premises_and_conclusion(question: str, response: str) -> Dict[str, Any]:
    prompt = f"""Your job is to classify the reasoning of an LLM response to a binary classification task. The response will be in the form of a chain of though, with a final answer. Ultimately, our goal is to use your classifications to taxonomize the LLM's reasoning patterns.
    
For the given response, you must:
1. Extract all premises (statements used as evidence or assumptions). Try to break these up into discrete statements or facts. 
2. Identify the conclusion (the main claim or result). Usually will be of the form, "the best answer is: ...".
3. Evaluate whether each premise is factually true or false.
4. Determine if the conclusion logically follows from the premises (assuming the premises are true). This can be difficult, particularly when the premises are false. Keep in mind that your job is to only classify whether the conclusion logically follows from the premises the LLM stated, not whether or not the conclusion itself is true.
5. Set refuse to true ONLY if you cannot complete the analysis (e.g., the text is incoherent, you cannot parse the premises, the answer is unclear, etc.)

Important:
- Premises should be distinct, atomic statements.
- The conclusion should be the final claim or answer.
- For 'follows_from_premises': evaluate pure logical validity assuming premises are true, regardless of their actual truth.
- Set refuse to false for normal analysis, true only if unable to analyze.

You will also be given the prompt the LLM is responding to. This will help you understand the LLM's reasoning.

A common question format is: "Is the following statement plausible: <statement>". In order to evaluate the LLM's reasoning, you will have to consider the response in the context of the question.

Here is the question:
{question}

Here is the response:
{response}
"""


    resp = client.responses.parse(
        model="gpt-5",
        input=[
            {"role": "user", "content": prompt},
        ],
        text_format=LogicalAnalysis,
    )

    event = resp.output_parsed

    model_obj = LogicalAnalysis.model_validate(event)  # pydantic v2

    return _model_to_dict(model_obj)

def is_non_entailment(analysis: Dict[str, Any]) -> bool:
    all_premises_true = all(analysis["premise_truth_values"])
    return all_premises_true and (not analysis["follows_from_premises"])

def is_confabulation(analysis: Dict[str, Any]) -> bool:
    any_premises_false = not all(analysis["premise_truth_values"])
    return any_premises_false and analysis["follows_from_premises"]

def is_hallucination(analysis: Dict[str, Any]) -> bool:
    pts = analysis.get("premise_truth_values", [])
    any_premises_false = len(pts) > 0 and (not all(pts))
    return any_premises_false and (not analysis["follows_from_premises"])

def is_sound(analysis: Dict[str, Any]) -> bool:
    all_premises_true = all(analysis["premise_truth_values"])
    return all_premises_true and analysis["follows_from_premises"]

def is_refuse(analysis: Dict[str, Any]) -> bool:
    return analysis["refuse"]


## Example Usage

In [5]:
# # Example: Test the classification function
# example_question = "Is the following statement plausible: Penguins can fly"

# example_response = """
# Let me analyze this step by step.

# First, all birds have wings. Second, penguins are birds. 
# Third, things with wings can typically fly.

# Therefore, penguins can fly.
# """

# result = classify_premises_and_conclusion(example_question, example_response)
# print(is_non_entailment(result))
# print(is_confabulation(result))
# print(is_hallucination(result))
# print(is_sound(result))
# print(is_refuse(result))
# print(json.dumps(result, indent=4))

## Load and Process Experiment Results

In [6]:
import pickle
from pathlib import Path

def load_experiment_responses(model_name: str, dataset_name: str, split: str = "test") -> List[Dict[str, Any]]:
    """
    Load model responses from experiment cache.
    
    Args:
        model_name: Name of the model (e.g., 'google_gemma-2-2b-it')
        dataset_name: Name of the dataset (e.g., 'logical_deduction')
        split: 'train' or 'test'
        
    Returns:
        List of response dictionaries
    """
    cache_path = Path(CACHE_DIR) / model_name / dataset_name
    
    # Find the split directory
    split_dirs = list(cache_path.glob("split_*"))
    if not split_dirs:
        print(f"No split directories found for {model_name}/{dataset_name}")
        return []
    
    # Use the first split directory
    split_dir = split_dirs[0]
    
    # Find experiment directories
    exp_dirs = list(split_dir.glob("*"))
    if not exp_dirs:
        print(f"No experiment directories found in {split_dir}")
        return []
    
    # Use the first experiment directory
    exp_dir = exp_dirs[0]
    
    # Load the generations file
    gen_file = exp_dir / "data" / f"{split}_generations.pkl"
    if not gen_file.exists():
        print(f"Generations file not found: {gen_file}")
        return []
    
    with open(gen_file, 'rb') as f:
        data = pickle.load(f)
    
    return data

# Example: Load responses
# responses = load_experiment_responses("google_gemma-2-2b-it", "logical_deduction")
# print(f"Loaded {len(responses)} responses")

In [7]:
def batch_classify_responses(responses: List[Dict[str, Any]], max_samples: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Classify multiple responses in batch.
    
    Args:
        responses: List of response dictionaries from experiments
        max_samples: Maximum number of samples to process (None for all)
        
    Returns:
        List of classification results
    """
    results = []
    
    # Limit samples if specified
    if max_samples:
        responses = responses[:max_samples]
    
    for i, response_data in enumerate(responses):
        print(f"Processing response {i+1}/{len(responses)}...")
        
        # Extract the response text (adjust key based on actual data structure)
        if isinstance(response_data, dict):
            response_text = response_data.get('response', '') or response_data.get('text', '')
        else:
            response_text = str(response_data)
        
        # Classify the response with question context if available
        q_raw = response_data.get('prompt') if isinstance(response_data, dict) else None
        if isinstance(response_data, dict) and q_raw is None:
            q_raw = response_data.get('original_prompt') or response_data.get('question')
        q_text = None
        if q_raw is not None:
            try:
                q_text = parse_question(str(q_raw))
            except Exception:
                q_text = str(q_raw)
        classification = classify_premises_and_conclusion(q_text or "", response_text)
        
        # Add metadata
        classification['response_index'] = i
        classification['original_response'] = response_text
        
        results.append(classification)
    
    return results

# Example usage:
# classifications = batch_classify_responses(responses, max_samples=5)
# for cls in classifications:
#     print(f"\nResponse {cls['response_index']}:")
#     print(f"  Premises: {cls['premises']}")
#     print(f"  Conclusion: {cls['conclusion']}")
#     print(f"  Follows from premises: {cls['follows_from_premises']}")

## Save Results

In [8]:
def save_classifications(classifications: List[Dict[str, Any]], output_path: str):
    """
    Save classification results to JSON file.
    
    Args:
        classifications: List of classification results
        output_path: Path to save the JSON file
    """
    with open(output_path, 'w') as f:
        json.dump(classifications, f, indent=2)
    print(f"Saved {len(classifications)} classifications to {output_path}")

# Example:
# save_classifications(classifications, "classification_results.json")

In [17]:

from pathlib import Path
import pickle
import json
import random
import re
import time
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

# Reuse OpenAI client and classifier from above notebook cells
# classify_premises_and_conclusion(response_text: str) -> Dict[str, Any]

STEERING_RE = re.compile(r"^steering_alpha_([+-]?[0-9]+(?:\.[0-9]+)?)_(yes|no)\.pkl$")


def _classify_with_retries(question_text: str, response_text: str, retries: int = 3, backoff: float = 1.5):
    last_err = None
    for attempt in range(retries):
        try:
            return classify_premises_and_conclusion(question_text, response_text), None
        except Exception as e:
            last_err = str(e)
            if attempt < retries - 1:
                time.sleep(backoff ** attempt)
    return None, last_err


def _parse_alpha_label(filename: str) -> Optional[Tuple[float, str]]:
    m = STEERING_RE.match(filename)
    if not m:
        return None
    alpha_str, label = m.group(1), m.group(2)
    try:
        alpha = float(alpha_str)
    except ValueError:
        return None
    return alpha, label


def _compute_direction(alpha: float, filename: str) -> Optional[str]:
    # Match repo logic for direction inference
    if alpha == 0:
        if filename.endswith("_yes.pkl"):
            return "no"
        if filename.endswith("_no.pkl"):
            return "yes"
        return None
    return "yes" if alpha > 0 else "no"


def _normalize_generation(g: Any) -> str:
    if isinstance(g, list) and len(g) == 1 and isinstance(g[0], str):
        return g[0]
    return str(g)


def _normalize_prompt(p: Any) -> str:
    if isinstance(p, list) and all(isinstance(x, dict) and "content" in x for x in p):
        parts = []
        for msg in p:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            parts.append(f"{role}: {content}")
        return "\n".join(parts)
    return str(p)


def _read_existing_indices(jsonl_path: Path) -> Set[int]:
    indices: Set[int] = set()
    if not jsonl_path.exists():
        return indices
    with jsonl_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                idx = int(obj.get("sample_idx", -1))
                if idx >= 0:
                    indices.add(idx)
            except Exception:
                continue
    return indices


def _count_existing_for_group(root_out: Path, model: str, dataset: str, alpha_abs: float, direction: str) -> int:
    base = root_out / model / dataset
    if not base.exists():
        return 0
    count = 0
    for split_dir in base.glob("split_*/"):
        for exp_dir in split_dir.iterdir():
            steering_dir = exp_dir / "steering"
            if not steering_dir.exists():
                continue
            for jsonl_file in steering_dir.glob("steering_alpha_*_*.jsonl"):
                parsed = _parse_alpha_label(jsonl_file.stem + ".pkl")
                if not parsed:
                    continue
                a, _ = parsed
                dirn = _compute_direction(a, jsonl_file.name.replace(".jsonl", ".pkl"))
                if dirn == direction and abs(abs(a) - alpha_abs) < 1e-9:
                    with jsonl_file.open("r") as f:
                        for _ in f:
                            count += 1
    return count


def classify_steering_cache(
    root_in: str = "darius_cache/experiments",
    root_out: str = "darius_cache/rollout_classification",
    max_gen: Optional[int] = None,
    seed: int = 42,
    overwrite: bool = False,
    model_filters: Optional[List[str]] = None,
    dataset_filters: Optional[List[str]] = None,
    workers: int = 4,
    retries: int = 3,
):
    random.seed(seed)
    root_in_p = Path(root_in)
    root_out_p = Path(root_out)
    root_out_p.mkdir(parents=True, exist_ok=True)

    # Count total models and datasets for progress tracking
    all_models = [p.name for p in root_in_p.iterdir() if p.is_dir()]
    if model_filters:
        all_models = [m for m in all_models if m in model_filters]
    
    print(f"🚀 Starting classification across {len(all_models)} models...")
    print(f"📂 Input: {root_in}")
    print(f"📁 Output: {root_out}")
    print(f"🎯 Max per group: {max_gen if max_gen else 'unlimited'}")
    if model_filters:
        print(f"🔍 Model filters: {model_filters}")
    if dataset_filters:
        print(f"🔍 Dataset filters: {dataset_filters}")
    print("=" * 60)

    # Track totals per group
    group_totals = {}  # (model, dataset, alpha_abs, direction) -> count
    
    model_count = 0
    for model_path in sorted(p for p in root_in_p.iterdir() if p.is_dir()):
        model_name = model_path.name
        if model_filters and model_name not in model_filters:
            continue

        model_count += 1
        all_datasets = [p.name for p in model_path.iterdir() if p.is_dir()]
        if dataset_filters:
            all_datasets = [d for d in all_datasets if d in dataset_filters]
        
        print(f"\n📊 [{model_count}/{len(all_models)}] Processing model: {model_name}")
        print(f"   Datasets to process: {len(all_datasets)}")

        dataset_count = 0
        for dataset_path in sorted(p for p in model_path.iterdir() if p.is_dir()):
            dataset_name = dataset_path.name
            if dataset_filters and dataset_name not in dataset_filters:
                continue

            dataset_count += 1
            all_splits = [p.name for p in dataset_path.iterdir() if p.is_dir() and p.name.startswith("split_")]
            print(f"   📁 [{dataset_count}/{len(all_datasets)}] Dataset: {dataset_name} ({len(all_splits)} splits)")

            split_count = 0
            for split_path in sorted(p for p in dataset_path.iterdir() if p.is_dir() and p.name.startswith("split_")):
                split_count += 1
                all_exps = [p.name for p in split_path.iterdir() if p.is_dir()]
                print(f"      📂 [{split_count}/{len(all_splits)}] Split: {split_path.name} ({len(all_exps)} experiments)")

                exp_count = 0
                for exp_path in sorted(p for p in split_path.iterdir() if p.is_dir()):
                    steering_dir = exp_path / "steering"
                    if not steering_dir.exists():
                        continue

                    exp_count += 1
                    all_pkl_files = list(steering_dir.glob("steering_alpha_*_*.pkl"))
                    print(f"         🧪 [{exp_count}/{len(all_exps)}] Experiment: {exp_path.name} ({len(all_pkl_files)} steering files)")

                    out_dir = Path(str(steering_dir).replace(str(root_in_p), str(root_out_p)))
                    out_dir.mkdir(parents=True, exist_ok=True)

                    pkl_count = 0
                    for pkl_file in sorted(steering_dir.glob("steering_alpha_*_*.pkl")):
                        pkl_count += 1
                        parsed = _parse_alpha_label(pkl_file.name)
                        if not parsed:
                            print(f"            ⚠️  [{pkl_count}/{len(all_pkl_files)}] Skipping {pkl_file.name} (parse failed)")
                            continue
                        alpha, label = parsed
                        direction = _compute_direction(alpha, pkl_file.name)
                        if direction is None:
                            print(f"            ⚠️  [{pkl_count}/{len(all_pkl_files)}] Skipping {pkl_file.name} (no direction)")
                            continue
                        alpha_abs = abs(alpha)

                        print(f"            🎯 [{pkl_count}/{len(all_pkl_files)}] Processing {pkl_file.name} (α={alpha}, dir={direction})")

                        out_jsonl = out_dir / (pkl_file.stem + ".jsonl")
                        existing_idx: Set[int] = set()
                        if out_jsonl.exists() and not overwrite:
                            existing_idx = _read_existing_indices(out_jsonl)
                            if existing_idx:
                                print(f"               📝 Found {len(existing_idx)} existing classifications")

                        # Enforce per-group quota
                        quota_remaining = None
                        if max_gen is not None:
                            existing_group = _count_existing_for_group(root_out_p, model_name, dataset_name, alpha_abs, direction)
                            quota_remaining = max(max_gen - existing_group, 0)
                            print(f"               📊 Group quota: {existing_group}/{max_gen} used, {quota_remaining} remaining")
                            if quota_remaining == 0:
                                print(f"               ⏭️  Skipping - quota exhausted")
                                continue

                        # Read pickle
                        print(f"               📖 Loading pickle file...")
                        with pkl_file.open("rb") as f:
                            data = pickle.load(f)
                        
                        if not isinstance(data, list):
                            print(f"               ❌ Error: Expected list, got {type(data)}")
                            raise ValueError(f"Expected list, got {type(data)}")

                        # Successful-only
                        candidates: List[Tuple[int, Dict[str, Any]]] = [
                            (i, s) for i, s in enumerate(data) if s.get("success", False) is True
                        ]
                        print(f"               ✅ Found {len(candidates)} successful samples (out of {len(data)} total)")

                        if existing_idx and not overwrite:
                            candidates = [(i, s) for (i, s) in candidates if i not in existing_idx]
                            print(f"               🔄 After filtering existing: {len(candidates)} candidates remain")
                        
                        if not candidates:
                            print(f"               ⚠️  No candidates found - skipping")
                            continue

                        if quota_remaining is not None:
                            n = min(len(candidates), quota_remaining)
                            selected = random.sample(candidates, n) if len(candidates) > n else candidates
                            print(f"               🎲 Selected {len(selected)} samples for classification")
                        else:
                            selected = candidates
                            print(f"               📝 Will classify all {len(selected)} candidates")

                        # Build records in parallel
                        def _build_record(sample_idx: int, s: Dict[str, Any]) -> Dict[str, Any]:
                            prompt_raw = s.get("original_prompt", "")
                            steered = s.get("steered_generation", "")
                            prompt_text = _normalize_prompt(prompt_raw)
                            response_text = _normalize_generation(steered)
                            
                            # Parse question and classify with retries
                            try:
                                question_text = parse_question(prompt_text)
                            except Exception:
                                question_text = prompt_text
                            
                            classification, cls_err = _classify_with_retries(question_text, response_text, retries=retries)
                            
                            # Compute per-sample labels
                            labels: List[str] = []
                            ne = cf = hal = snd = ref = False
                            if classification:
                                ne = is_non_entailment(classification)
                                cf = is_confabulation(classification)
                                hal = is_hallucination(classification)
                                snd = is_sound(classification)
                                ref = is_refuse(classification)
                                if ne: labels.append("non_entailment")
                                if cf: labels.append("confabulation")
                                if hal: labels.append("hallucination")
                                if snd: labels.append("sound")
                                if ref: labels.append("refuse")

                            return {
                                "model": model_name,
                                "dataset": dataset_name,
                                "split": split_path.name,
                                "experiment_hash": exp_path.name,
                                "alpha": alpha,
                                "alpha_abs": alpha_abs,
                                "label": label,
                                "direction": direction,
                                "sample_idx": sample_idx,
                                "prompt": prompt_raw,
                                "prompt_text": prompt_text,
                                "steered_generation": response_text,
                                "original_answer": s.get("original_answer"),
                                "new_answer": s.get("new_answer"),
                                "target_answer": s.get("target_answer"),
                                "original_letter": s.get("original_letter"),
                                "new_letter": s.get("new_letter"),
                                "category": s.get("category"),
                                "success": s.get("success"),
                                "is_valid_parse": s.get("is_valid_parse"),
                                "classification": classification,
                                "classification_error": cls_err,
                                "non_entailment": ne,
                                "confabulation": cf,
                                "hallucination": hal,
                                "sound": snd,
                                "refuse_flag": ref,
                                "labels": labels,
                            }

                        print(f"               🔄 Processing {len(selected)} samples with {workers} workers...")
                        records: List[Dict[str, Any]] = []
                        stats = {"non_entailment": 0, "confabulation": 0, "hallucination": 0, "sound": 0, "refuse": 0}
                        total = 0
                        
                        with ThreadPoolExecutor(max_workers=max(1, int(workers))) as ex:
                            futures = [ex.submit(_build_record, idx, s) for idx, s in selected]
                            for sample_num, fut in enumerate(as_completed(futures), 1):
                                try:
                                    rec = fut.result()
                                    records.append(rec)
                                    
                                    # Update stats and show progress
                                    if rec["classification"]:
                                        total += 1
                                        stats["non_entailment"] += int(rec["non_entailment"])
                                        stats["confabulation"] += int(rec["confabulation"])
                                        stats["hallucination"] += int(rec["hallucination"])
                                        stats["sound"] += int(rec["sound"])
                                        stats["refuse"] += int(rec["refuse_flag"])
                                    
                                    labels = rec["labels"]
                                    print(f"               🔍 [{sample_num}/{len(selected)}] Sample {rec['sample_idx']}: {','.join(labels) if labels else 'none'}")
                                    
                                except Exception as e:
                                    print(f"               ❌ [{sample_num}/{len(selected)}] Failed to process sample: {e}")
                                    records.append({
                                        "model": model_name,
                                        "dataset": dataset_name,
                                        "split": split_path.name,
                                        "experiment_hash": exp_path.name,
                                        "alpha": alpha,
                                        "alpha_abs": alpha_abs,
                                        "label": label,
                                        "direction": direction,
                                        "sample_idx": -1,
                                        "classification": None,
                                        "classification_error": str(e),
                                    })

                        # Sort records by sample_idx for deterministic output
                        records.sort(key=lambda r: r.get("sample_idx", -1))

                        mode = "w" if overwrite else ("a" if out_jsonl.exists() else "w")
                        print(f"               💾 Writing to {out_jsonl.name} (mode: {mode})")
                        with out_jsonl.open(mode) as w:
                            for rec in records:
                                w.write(json.dumps(rec) + "\n")

                            # Print legible stats per steering file
                            if total > 0:
                                pct = {k: f"{(v/total)*100:.1f}%" for k, v in stats.items()}
                                print(f"               📈 Final stats: total={total} | "
                                      f"non_entailment={stats['non_entailment']} ({pct['non_entailment']}), "
                                      f"confabulation={stats['confabulation']} ({pct['confabulation']}), "
                                      f"hallucination={stats['hallucination']} ({pct['hallucination']}), "
                                      f"sound={stats['sound']} ({pct['sound']}), "
                                      f"refuse={stats['refuse']} ({pct['refuse']})")
                            else:
                                print(f"               ⚠️  No successful classifications in this file")

                            # Track group totals
                            group_key = (model_name, dataset_name, alpha_abs, direction)
                            if group_key not in group_totals:
                                group_totals[group_key] = 0
                            group_totals[group_key] += total

    # Print summary report
    print("\n" + "=" * 80)
    print("📊 CLASSIFICATION SUMMARY")
    print("=" * 80)
    
    if group_totals:
        total_classified = sum(group_totals.values())
        print(f"🎯 Total samples classified: {total_classified}")
        print(f"🔢 Unique (model, dataset, α, direction) groups: {len(group_totals)}")
        print("\nPer-group breakdown:")
        
        for (model, dataset, alpha_abs, direction), count in sorted(group_totals.items()):
            quota_str = f"/{max_gen}" if max_gen else ""
            print(f"  📈 {model}/{dataset} |α|={alpha_abs} dir={direction}: {count}{quota_str} samples")
    else:
        print("⚠️  No samples were classified!")
    
    print("\n🎉 Classification complete!")



In [ ]:
# Example: run batch classification over a subset
# Only successful steering examples are classified. Set max_gen per (model,dataset,|alpha|,direction).
classify_steering_cache(max_gen=20, workers=8, retries=1)


🚀 Starting classification across 8 models...
📂 Input: darius_cache/experiments
📁 Output: darius_cache/rollout_classification
🎯 Max per group: 20

📊 [1/8] Processing model: Qwen_Qwen2.5-1.5B-Instruct
   Datasets to process: 4
   📁 [1/4] Dataset: anachronisms (1 splits)
      📂 [1/1] Split: split_42_500_500 (1 experiments)
         🧪 [1/1] Experiment: 60af4fd1aece (22 steering files)
            🎯 [1/22] Processing steering_alpha_-10_yes.pkl (α=-10.0, dir=no)
               📝 Found 20 existing classifications
               📊 Group quota: 20/20 used, 0 remaining
               ⏭️  Skipping - quota exhausted
            🎯 [2/22] Processing steering_alpha_-12_yes.pkl (α=-12.0, dir=no)
               📝 Found 20 existing classifications
               📊 Group quota: 20/20 used, 0 remaining
               ⏭️  Skipping - quota exhausted
            🎯 [3/22] Processing steering_alpha_-14_yes.pkl (α=-14.0, dir=no)
               📝 Found 20 existing classifications
               📊 Group quota: 2